In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns

from scipy.stats import norm
import scipy.stats

In [16]:
#Load the data set
iris = sns.load_dataset("iris")
iris = iris.rename(index = str, columns = {'sepal_length':'1_sepal_length','sepal_width':'2_sepal_width', 'petal_length':'3_petal_length', 'petal_width':'4_petal_width'})

df1 = iris[["1_sepal_length", "2_sepal_width",'species']]

In [3]:
df1

,1_sepal_length,2_sepal_width,species
0,5.1,3.5,setosa
1,4.9,3.0,setosa
2,4.7,3.2,setosa
3,4.6,3.1,setosa
4,5.0,3.6,setosa
...,...,...,...
145,6.7,3.0,virginica
146,6.3,2.5,virginica
147,6.5,3.0,virginica
148,6.2,3.4,virginica


In [17]:
#Estimating the parameters
mu_list = np.split(df1.groupby('species').mean().values,[1,2])
std_list = np.split(df1.groupby('species').std().values,[1,2], axis = 0)
pi_list = df1.iloc[:,2].value_counts().values / len(df1)

In [5]:
pi_list

array([0.33333333, 0.33333333, 0.33333333])

In [6]:
mu_list

[array([[5.006, 3.428]]), array([[5.936, 2.77 ]]), array([[6.588, 2.974]])]

In [18]:
def predict_NB_gaussian_class(X,mu_list,std_list,pi_list):
#Returns the class for which the Gaussian Naive Bayes objective function has greatest value
    scores_list = []
    classes = len(mu_list)
    for p in range(classes):
        score = (norm.pdf(x = X[0], loc = mu_list[p][0][0], scale = std_list[p][0][0] )
        * norm.pdf(x = X[1], loc = mu_list[p][0][1], scale = std_list[p][0][1] )
        * pi_list[p])
        scores_list.append(score)
        
    return np.argmax(scores_list)

In [19]:
X = np.array(df1.iloc[:, 0:2])

In [20]:
y_labels = df1.iloc[:,2].replace({'setosa':0,'versicolor':1,'virginica':2}).copy()

C:\Users\WinIF Chung\AppData\Local\Temp\ipykernel_1964\312446389.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  y_labels = df1.iloc[:,2].replace({'setosa':0,'versicolor':1,'virginica':2}).copy()


In [10]:
 y_labels

0      0
1      0
2      0
3      0
4      0
      ..
145    2
146    2
147    2
148    2
149    2
Name: species, Length: 150, dtype: int64

In [21]:
y_pred = np.zeros(X.shape[0])
for i in range(X.shape[0]):
    y_pred[i] = predict_NB_gaussian_class( X[i], mu_list, std_list, pi_list)

print(y_pred)

[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0.
 0. 0. 2. 2. 2. 1. 2. 1. 2. 1. 2. 1. 1. 1. 1. 1. 1. 2. 1. 1. 1. 1. 1. 1.
 1. 1. 2. 2. 2. 2. 1. 1. 1. 1. 1. 1. 1. 2. 2. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 2. 1. 2. 1. 2. 2. 1. 2. 2. 2. 2. 1. 2. 1. 1. 2. 2. 2. 2. 1.
 2. 1. 2. 1. 2. 2. 1. 1. 1. 2. 2. 2. 1. 1. 1. 2. 2. 2. 1. 2. 2. 2. 1. 2.
 2. 2. 1. 2. 2. 1.]


In [22]:
from sklearn.metrics import accuracy_score, precision_score, recall_score
accuracy_score(y_pred, y_labels)


0.78

In [23]:
# Đọc tệp dữ liệu breast-cancer-wisconsin.data
col_names = ['id', 'clump_thickness', 'cell_size_uniformity', 'cell_shape_uniformity',
             'marginal_adhesion', 'single_epithelial_size', 'bare_nuclei',
             'bland_chromatin', 'normal_nucleoli', 'mitoses', 'class']
df = pd.read_csv('breast-cancer-wisconsin.data', header=None, names=col_names)

# Tiền xử lý: Xử lý 16 giá trị thiếu '?' trong bare_nuclei bằng trung vị (median)
df['bare_nuclei'] = pd.to_numeric(df['bare_nuclei'].replace('?', np.nan))
df['bare_nuclei'] = df.groupby('class')['bare_nuclei'].transform(lambda s: s.fillna(s.median()))

print('5 dòng đầu của dữ liệu sau xử lý:')
print(df.head())

# Tách dữ liệu Test: 80 mẫu lành tính (class 2) và 40 mẫu ác tính (class 4)
test_benign = df[df['class'] == 2].sample(80, random_state=42)
test_malignant = df[df['class'] == 4].sample(40, random_state=42)
test_df = pd.concat([test_benign, test_malignant])

# Dữ liệu Training là phần còn lại (579 mẫu)
train_df = df.drop(test_df.index)
print(f'Tập Train: {train_df.shape[0]} mẫu, Tập Test: {test_df.shape[0]} mẫu')


   1000025  5   1  1.1  1.2  2 1.3  3  1.4  1.5  2.1
0  1002945  5   4    4    5  7  10  3    2    1    2
1  1015425  3   1    1    1  2   2  3    1    1    2
2  1016277  6   8    8    1  3   4  3    7    1    2
3  1017023  4   1    1    3  2   1  3    1    1    2
4  1017122  8  10   10    8  7  10  9    7    1    4


In [ ]:
# Huấn luyện mô hình GaussianNB và đánh giá trên tập Test
feature_cols = [c for c in col_names if c not in ['id', 'class']]
X_train = train_df[feature_cols].values
y_train = train_df['class'].values

X_test = test_df[feature_cols].values
y_test = test_df['class'].values

model_cancer = GaussianNB()
model_cancer.fit(X_train, y_train)
y_pred_cancer = model_cancer.predict(X_test)

print(f'Độ chính xác (Accuracy): {accuracy_score(y_test, y_pred_cancer)*100:.2f}%')
print(f'Precision (Lớp ác tính 4): {precision_score(y_test, y_pred_cancer, pos_label=4)*100:.2f}%')
print(f'Recall (Lớp ác tính 4):    {recall_score(y_test, y_pred_cancer, pos_label=4)*100:.2f}%')
print(f'F1-Score (Lớp ác tính 4):  {f1_score(y_test, y_pred_cancer, pos_label=4)*100:.2f}%')
print('\nBáo cáo phân loại chi tiết:\n', classification_report(y_test, y_pred_cancer, target_names=['Lành tính (2)', 'Ác tính (4)']))


In [ ]:
# BÀI TẬP MỞ RỘNG: PHÂN LOẠI HOA IRIS VỚI ĐỦ 4 THUỘC TÍNH (TRAIN:TEST = 4:1)
from sklearn.model_selection import train_test_split

X_iris = iris[['1_sepal_length', '2_sepal_width', '3_petal_length', '4_petal_width']].values
y_iris = iris['species'].replace({'setosa': 0, 'versicolor': 1, 'virginica': 2}).values

X_tr, X_te, y_tr, y_te = train_test_split(X_iris, y_iris, test_size=0.2, random_state=42, stratify=y_iris)
gnb_iris = GaussianNB()
gnb_iris.fit(X_tr, y_tr)
y_pred_iris = gnb_iris.predict(X_te)

print('Độ chính xác Iris với 4 đặc trưng:', accuracy_score(y_te, y_pred_iris))
print('\nClassification Report:\n', classification_report(y_te, y_pred_iris, target_names=['setosa', 'versicolor', 'virginica']))
